In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
!pip install gradio

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import gradio as gr
import matplotlib.pyplot as plt


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
DATA_DIR = "/content/drive/MyDrive/flood_detection_project/data/processed"

with open(os.path.join(DATA_DIR, "test.pkl"), "rb") as f:
    test_data = pickle.load(f)

print("Test samples:", len(test_data))


Test samples: 41


In [ ]:
def center_crop_to_match(enc_feat, dec_feat):
    _, _, h, w = dec_feat.shape
    enc_h, enc_w = enc_feat.shape[2], enc_feat.shape[3]
    sh = (enc_h - h) // 2
    sw = (enc_w - w) // 2
    return enc_feat[:, :, sh:sh+h, sw:sw+w]


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        e3c = center_crop_to_match(e3, d3)
        d3 = self.dec3(torch.cat([d3, e3c], dim=1))

        d2 = self.up2(d3)
        e2c = center_crop_to_match(e2, d2)
        d2 = self.dec2(torch.cat([d2, e2c], dim=1))
        d1 = self.up1(d2)
        e1c = center_crop_to_match(e1, d1)
        d1 = self.dec1(torch.cat([d1, e1c], dim=1))

        return self.out(d1)


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/flood_detection_project/models/unet_30epochs.pth"

model = UNet(in_channels=4).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("Model loaded")


Model loaded


In [ ]:
def run_demo(sample_index):
    X, y = test_data[int(sample_index)]
    X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(X_tensor)
        prob = torch.sigmoid(logits).squeeze().cpu().numpy()

    # Severity zoning
    severity = np.zeros_like(prob)
    severity[(prob >= 0.3) & (prob < 0.6)] = 1
    severity[prob >= 0.6] = 2

    # Plot probability
    fig1, ax1 = plt.subplots()
    im1 = ax1.imshow(prob, cmap="jet")
    fig1.colorbar(im1, ax=ax1)
    ax1.set_title("Flood Probability Map")
    ax1.axis("off")

    # Plot severity
    fig2, ax2 = plt.subplots()
    im2 = ax2.imshow(severity, cmap="viridis")
    fig2.colorbar(im2, ax=ax2, ticks=[0,1,2])
    ax2.set_title("Flood Severity Zones")
    ax2.axis("off")

    return fig1, fig2


In [ ]:
sample_indices = list(range(len(test_data)))

demo = gr.Interface(
    fn=run_demo,
    inputs=gr.Dropdown(sample_indices, label="Select Test Sample"),
    outputs=[
        gr.Plot(label="Flood Probability"),
        gr.Plot(label="Flood Severity")
    ],
    title="Flood Detection & Severity Mapping Demo",
    description="Prototype UI demonstrating flood probability and severity zoning using Sentinel-1 SAR data.",
    allow_flagging='never'
)

demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a836c8a601f0911093.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
